# Clef decision training with `DecisionTrainer` and Accelerate

This notebook trains a Clef decision model with TRL's experimental `DecisionTrainer`, on one GPU or on several with
Accelerate, and then lets you use it. In order:

1. build an environment with a known torch and the two forks, pinned;
2. write one configuration and train with cross-entropy (`accelerate launch`, in fresh processes);
3. continue that model with an RLCD phase (cross-entropy first, then RLCD);
4. read the metrics of both runs;
5. ask the trained model questions;
6. optionally: convert an older checkpoint, or time training steps.

Run it on a Linux machine with an NVIDIA GPU. With `QUICK = True` (the default) each stage is a short run on a
sample, so the whole notebook takes minutes; set `QUICK = False` for the full recipe.

By default the notebook makes its own virtual environment inside the run folder, so the Python packages already on
the machine (an older torch, a NumPy that torch was built against) are neither used nor changed. This kernel needs
nothing but the standard library: everything that touches torch runs in fresh processes of that environment.

## Settings

This is the only cell to edit.

In [ ]:
from pathlib import Path

# Where the code comes from: your forks, pinned to the commits this notebook was written against
TRANSFORMERS_SOURCE = "https://github.com/ericlewis/transformers/archive/d497f14dfda0ac54ba8aff55cdf1275b69f7026a.tar.gz"
TRL_SOURCE = "https://github.com/ericlewis/trl/archive/266d7b05ca2cdb1eed11de136f4a3202c60c932b.tar.gz"
FLA_SOURCE = "https://github.com/fla-org/flash-linear-attention/archive/9c102739e348968753d86bb729c57dd6efa1c413.tar.gz"

# Where it runs
ENVIRONMENT = "own"  # "own": a virtual environment in the run folder, with its own torch. "kernel": this kernel's Python
TORCH = "torch==2.14.1"  # for "own"; the build for this machine's GPU and driver is picked automatically
INSTALL = True  # False: use the environment as it is

# What to train
MODEL = "MinimaLabs/KLM-1.6"
MODEL_REVISION = "8fabc98abe194f7450cf19ed743caa7e85a4e5b0"
DATASET = "LocalLLaMA/typed-decisions"  # rows with `state`, `questions` and `gold`
DATASET_CONFIG = "all"
DATASET_REVISION = None
EVAL_SPLIT = None  # None: hold out 5% of the training split. Or the name of a split, such as "validation"
HEAD = "cosine"  # "cosine" (small) or "joint" (Clef's original head)
MODE = "full"  # "full", "head" (backbone frozen) or "lora"

# How to run it
QUICK = True  # True: a short run on a sample, to see every stage work. False: the full recipe
NUM_GPUS = None  # None: every visible GPU (CUDA_VISIBLE_DEVICES is honoured)
GLOBAL_BATCH_SIZE = 32  # records per optimizer step, split evenly across the GPUs
EPOCHS = 3
LEARNING_RATE = 1e-4  # backbone
HEAD_LEARNING_RATE = 3e-4
MAX_LENGTH = 2048  # tokens per prompt; a longer record has the end of its state cut
EVAL_EVERY_STEPS = 12
ACTIVATION_CHECKPOINTING = True  # less memory, slower steps
BACKBONE_ATTENTION = "kernels-community/flash-attn2@291664546de978ab2bb730554252e5e2c1099e4f"  # on CUDA; None: default
SEED = 3407

# After cross-entropy: the RLCD phase
RLCD_PHASE = True
RLCD_EPOCHS = 4
RLCD_SIGMAS = [0.4, 0.3, 0.2, 0.1]  # noise of each RLCD epoch
RLCD_WEIGHT = 0.25  # the loss of the phase is cross-entropy + RLCD_WEIGHT x the RLCD term

# Optional extras
EXISTING_CHECKPOINT = None  # a folder in the older layout (backbone + joint_head.safetensors) to convert and try
RUN_BENCHMARK = False  # time training steps on this machine
FLA_PRESET_DIR = None  # a folder of saved FLA kernel presets to reuse for every batch size (see the notes at the end)

WORK_DIR = Path("clef_decision_run").resolve()

## Environment

With `ENVIRONMENT = "own"` this makes a virtual environment in the run folder with [uv](https://docs.astral.sh/uv/),
installs torch for this machine's GPU, then the released packages for their dependencies, then the forks and FLA
(KLM's kernels) over them. Nothing outside the run folder is changed. The first build downloads a few GB of CUDA
libraries, so it can take several minutes; the output shows its progress.

`ENVIRONMENT = "kernel"` installs into this kernel's Python instead and keeps its torch. That torch must be 2.10 or
newer for the flash-attention kernel; where it was built against NumPy 1.x, NumPy 1.x is kept for it.

In [ ]:
import importlib.util
import json
import os
import shutil
import subprocess
import sys

WORK_DIR.mkdir(parents=True, exist_ok=True)
PACKAGES = ["transformers", "trl", "accelerate>=1.15.0", "datasets", "peft", "kernels", "einops", "pyyaml"]
FORKS = [TRANSFORMERS_SOURCE, TRL_SOURCE, FLA_SOURCE]


def run_environment():
    """The environment of a run: this one's, without anything left by another distributed job."""
    distributed = ("RANK", "LOCAL_RANK", "WORLD_SIZE", "LOCAL_WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT")
    environment = {k: v for k, v in os.environ.items() if not k.startswith("ACCELERATE_") and k not in distributed}
    environment["TOKENIZERS_PARALLELISM"] = "false"
    environment["PYTHONWARNINGS"] = "ignore"
    environment.setdefault("TRITON_CACHE_DIR", str(WORK_DIR / "cache" / "triton"))  # compiled kernels, kept between runs
    if FLA_PRESET_DIR is not None:
        environment["FLA_CACHE_MODE"] = "fuzzy"
        environment["FLA_CONFIG_DIR"] = str(Path(FLA_PRESET_DIR).expanduser().resolve())
    return environment


def stream(command):
    """Run a command in a fresh process, print its output as it comes, and stop the notebook if it fails."""
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    process = subprocess.Popen(
        command, cwd=WORK_DIR, env=run_environment(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"The command stopped with exit code {process.returncode}; its last lines are above")


def in_python(code, python=None):
    """Run a line of Python outside the kernel; returns (worked, what it printed, the end of its errors)."""
    done = subprocess.run([str(python or PY), "-c", code], capture_output=True, text=True)
    return done.returncode == 0, done.stdout.strip(), done.stderr.strip()[-1500:]


if ENVIRONMENT == "own":
    PY = WORK_DIR / "env" / "bin" / "python"
    if INSTALL or not PY.exists():
        if shutil.which("uv") is None and importlib.util.find_spec("uv") is None:
            stream([sys.executable, "-m", "pip", "install", "-q", "uv"])
        uv = [shutil.which("uv")] if shutil.which("uv") else [sys.executable, "-m", "uv"]
        if not PY.exists():
            stream([*uv, "venv", "--python", sys.executable, PY.parent.parent])
        install = [*uv, "pip", "install", "--python", PY, "--torch-backend=auto"]
        stream([*install, TORCH])  # a few GB of CUDA libraries the first time
        stream([*install, TORCH, *PACKAGES])  # torch named again, so nothing replaces the build just chosen
        stream([*install, "--no-deps", "--reinstall", *FORKS])
elif ENVIRONMENT == "kernel":
    PY = Path(sys.executable)
    if INSTALL:
        worked, torch_version, _ = in_python("import torch; print(torch.__version__)")
        if not worked:
            raise RuntimeError("torch is not importable in this kernel: install it first, or use ENVIRONMENT = 'own'")
        # A torch built against NumPy 1.x cannot use NumPy 2, and pyarrow 26 needs NumPy 2 without declaring it, so
        # pip sees neither conflict. Where NumPy is 1.x, or torch cannot read the NumPy 2 installed, stay on 1.x.
        _, numpy_version, _ = in_python("import importlib.metadata as m; print(m.version('numpy'))")
        reads_numpy, _, _ = in_python("import numpy, torch; torch.from_numpy(numpy.zeros(1))")
        keep_numpy_1 = ["numpy<2", "pyarrow>=24,<26"] if numpy_version.startswith("1.") or not reads_numpy else []
        if keep_numpy_1:
            print(f"torch {torch_version} with NumPy {numpy_version or 'missing'}: keeping NumPy 1.x for this torch")
        pip = [sys.executable, "-m", "pip", "install", "-q"]
        stream([*pip, *PACKAGES, *keep_numpy_1])
        stream([*pip, "--no-deps", "--force-reinstall", *FORKS])
else:
    raise ValueError("ENVIRONMENT must be 'own' or 'kernel'")

check = (
    "import accelerate, datasets, numpy, torch, transformers, trl; torch.from_numpy(numpy.zeros(1)); "
    "from transformers.loss.loss_utils import LOSS_MAPPING; "
    "assert 'ClefForDecision' in LOSS_MAPPING, 'this transformers has no Clef training support: is the fork installed?'; "
    "from trl.experimental.decision import DecisionTrainer, RlcdScheduleCallback; "
    "print('torch', torch.__version__, '| numpy', numpy.__version__, '| transformers', transformers.__version__, "
    "'| trl', trl.__version__, '| accelerate', accelerate.__version__)"
)
worked, versions, errors = in_python(check)
print(versions if worked else errors)
if not worked:
    raise RuntimeError("This environment cannot import what the notebook needs; see the message above")
print("Python of the runs:", PY)

## The example scripts and this machine

Training is done by the Clef example of the TRL fork, fetched here at the pinned commit. The GPUs are counted in a
separate process. The flash-attention kernel is tried once here: where it has no build for the machine's torch, the
backbone uses the default attention instead, and the cell says so.

In [ ]:
import io
import tarfile
import urllib.request

EXAMPLES = WORK_DIR / "trl"
TRAIN_SCRIPT = EXAMPLES / "examples/decisions/clef_typed_decisions.py"
BENCHMARK_SCRIPT = EXAMPLES / "examples/decisions/benchmark_clef.py"
if not TRAIN_SCRIPT.exists():
    with urllib.request.urlopen(TRL_SOURCE) as response:
        archive = tarfile.open(fileobj=io.BytesIO(response.read()), mode="r:gz")
    for member in archive.getmembers():
        parts = Path(member.name).parts[1:]  # without the top folder of the archive
        if member.isfile() and parts[:2] == ("examples", "decisions"):
            target = EXAMPLES.joinpath(*parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.extractfile(member).read())

probe = (
    "import json, torch; n = torch.cuda.device_count() if torch.cuda.is_available() else 0; "
    "print(json.dumps({'gpus': n, 'names': [torch.cuda.get_device_name(i) for i in range(n)], 'torch': torch.__version__}))"
)
worked, found, errors = in_python(probe)
if not worked:
    raise RuntimeError(errors)
machine = json.loads(found.splitlines()[-1])
ON_GPU = machine["gpus"] > 0
if NUM_GPUS is not None and NUM_GPUS > machine["gpus"]:
    raise ValueError(f"NUM_GPUS is {NUM_GPUS}, and this machine shows {machine['gpus']}")
PROCESSES = (NUM_GPUS or machine["gpus"]) if ON_GPU else 1
if ON_GPU and GLOBAL_BATCH_SIZE % PROCESSES:
    raise ValueError(f"GLOBAL_BATCH_SIZE {GLOBAL_BATCH_SIZE} does not divide evenly across {PROCESSES} GPUs")
PER_DEVICE_BATCH = GLOBAL_BATCH_SIZE // PROCESSES if ON_GPU else 2

ATTENTION = BACKBONE_ATTENTION if ON_GPU else None
if ATTENTION is not None and "/" in ATTENTION:  # a kernel from the Hub: does it have a build for this torch?
    trial = f"from transformers.modeling_flash_attention_utils import lazy_import_flash_attention as load; load({ATTENTION!r})"
    worked, _, errors = in_python(trial)
    if not worked:
        reason = next((line for line in reversed(errors.splitlines()) if line.strip()), "no reason given")
        print(f"The flash-attention kernel does not load here, so the backbone uses the default attention.\n  {reason[:300]}")
        ATTENTION = None

print("torch", machine["torch"], "|", machine["names"] or "no CUDA GPU: a tiny CPU run, only to check the stages")
print(f"{PROCESSES} process(es) x {PER_DEVICE_BATCH} records = {PROCESSES * PER_DEVICE_BATCH} records per step")
print("backbone attention:", ATTENTION or "default")

## The configuration

One file per run, with the same settings the example's own configuration uses (JSON, which YAML readers accept).
`write_config` is used again for the RLCD phase.

In [ ]:
def write_config(name, **changes):
    """Write `<name>.yaml` for a run whose output goes to `WORK_DIR/<name>`; `changes` override, `None` removes."""
    config = {
        "model_name_or_path": MODEL,
        "model_revision": MODEL_REVISION,
        "dataset_name": DATASET,
        "dataset_config": DATASET_CONFIG,
        "dataset_revision": DATASET_REVISION,
        "dataset_test_split": EVAL_SPLIT,
        "output_dir": str(WORK_DIR / name),
        "dtype": "float32",  # float32 weights under bf16 autocast, which KLM's fused kernels need
        "attn_implementation": ATTENTION,  # applied to the backbone only
        "mode": MODE,
        "head_type": HEAD,
        "max_length": MAX_LENGTH,
        "truncation": "state",
        "permute_fields": True,  # training batches reorder the questions; evaluation never does
        "per_device_train_batch_size": PER_DEVICE_BATCH,
        "per_device_eval_batch_size": 8 if ON_GPU else 2,
        "num_train_epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "head_learning_rate": HEAD_LEARNING_RATE,
        "optim": "adamw_torch_fused" if ON_GPU else "adamw_torch",
        "weight_decay": 0.05,
        "max_grad_norm": 1.0,
        "lr_scheduler_type": "cosine",
        "warmup_steps": 0.03,
        "activation_checkpointing": ACTIVATION_CHECKPOINTING and ON_GPU,
        "eval_strategy": "steps",
        "eval_steps": EVAL_EVERY_STEPS,
        "logging_steps": EVAL_EVERY_STEPS,
        "save_strategy": "best",  # a checkpoint at every evaluation that improves the metric below
        "save_total_limit": 1,
        "load_best_model_at_end": True,
        "metric_for_best_model": "eval_selection_score",
        "greater_is_better": True,
        "bf16": ON_GPU,
        "use_cpu": None if ON_GPU else True,
        "dataloader_num_workers": 4 if ON_GPU else 0,
        "disable_tqdm": True,
        "seed": SEED,
        "report_to": "none",
    }
    if QUICK and ON_GPU:  # 20 steps per epoch, 2 epochs
        config.update(max_train_samples=GLOBAL_BATCH_SIZE * 20, max_eval_samples=128, num_train_epochs=2, eval_steps=10, logging_steps=10)
    elif QUICK:  # a handful of steps on a CPU
        config.update(max_train_samples=16, max_eval_samples=8, max_steps=4, eval_steps=2, logging_steps=1)
    config.update(changes)
    config = {key: value for key, value in config.items() if value is not None}
    path = WORK_DIR / f"{name}.yaml"
    path.write_text(json.dumps(config, indent=2) + "\n")
    return path


cross_entropy_config = write_config("cross_entropy")
print(cross_entropy_config.read_text())

## Train with cross-entropy

`accelerate launch` starts one process per GPU (DDP) and streams their output here. `DecisionTrainer` scales the loss
and gathers the evaluation across processes; the first process tokenizes the dataset once and the others reuse it.

In [ ]:
def launch(config_path):
    command = [PY, "-m", "accelerate.commands.launch", "--num_processes", PROCESSES, "--num_machines", 1]
    command += ["--mixed_precision", "bf16" if ON_GPU else "no", "--dynamo_backend", "no"]
    if PROCESSES > 1:
        command += ["--multi_gpu", "--main_process_port", 0]
    if not ON_GPU:
        command += ["--cpu"]
    stream([*command, TRAIN_SCRIPT, "--config", config_path])


launch(cross_entropy_config)

## Continue with an RLCD phase

RLCD comes after cross-entropy: this run starts from the model just trained, with a fresh optimizer and lower
learning rates, and anneals the noise per epoch. Its loss keeps the cross-entropy term and adds `RLCD_WEIGHT` times
the RLCD term. A checkpoint keeps its own head, so `head_type` is left out.

In [ ]:
if RLCD_PHASE:
    rlcd_config = write_config(
        "rlcd",
        model_name_or_path=str(WORK_DIR / "cross_entropy"),
        model_revision=None,
        head_type=None,
        num_train_epochs=RLCD_EPOCHS,
        learning_rate=LEARNING_RATE / 5,
        head_learning_rate=HEAD_LEARNING_RATE / 5,
        rlcd_weight=RLCD_WEIGHT,
        rlcd_sigmas=RLCD_SIGMAS,
    )
    launch(rlcd_config)

## Metrics

The final evaluation of each run, on the same held-out records. `selection_score` is accuracy minus a quarter of the
KL divergence and a quarter of the Brier score; it is what picks the checkpoint that is kept.

In [ ]:
METRICS = ["accuracy", "soft_nll", "kl", "brier", "ece", "mean_confidence", "selection_score"]
runs = ["cross_entropy", *(["rlcd"] if RLCD_PHASE else [])]
results = {name: json.loads((WORK_DIR / name / "eval_results.json").read_text()) for name in runs}

print(f"{'':16}" + "".join(f"{name:>16}" for name in METRICS))
for name in runs:
    print(f"{name:16}" + "".join(f"{results[name]['eval_' + metric]:16.4f}" for metric in METRICS))
if QUICK:
    print("\nQUICK run: these numbers only show that the stages work.")

## Ask the trained model

The saved folder is a complete model: the decision pipeline loads it as it is. A request has a `state` and named
`questions` of type `noul` (yes or no), `choice` or `score`. `ask` runs the pipeline in a fresh process of the same
environment and returns one answer per request.

In [ ]:
ASK_SCRIPT = WORK_DIR / "ask.py"
ASK_SCRIPT.write_text('''
"""Answer the requests read from stdin with a saved decision model; the last line printed is the JSON result."""
import json
import sys

import torch
from transformers import pipeline

job = json.load(sys.stdin)
requests, gold = list(job["requests"]), None
if job.get("dataset"):
    from datasets import load_dataset

    row = load_dataset(job["dataset"]["name"], job["dataset"]["config"], revision=job["dataset"]["revision"], split=job["dataset"]["split"])[0]
    value = lambda field: json.loads(field) if isinstance(field, str) else field
    requests.insert(0, {"state": value(row["state"]), "questions": value(row["questions"])})
    gold = value(row["gold"])
on_cuda = torch.cuda.is_available()
decide = pipeline("decision", model=job["model"], dtype=torch.float32, device=0 if on_cuda else -1)
with torch.autocast("cuda", dtype=torch.bfloat16, enabled=on_cuda):
    answers = [decide(request) for request in requests]
print(json.dumps({"answers": answers, "gold": gold}, default=str))
''')

FINAL = WORK_DIR / ("rlcd" if RLCD_PHASE else "cross_entropy")


def ask(*requests, model=FINAL, dataset_row=False):
    """Answers of `model` to `requests`; with `dataset_row`, first to a record of the dataset, with its labels."""
    dataset = None
    if dataset_row:
        dataset = {"name": DATASET, "config": DATASET_CONFIG, "revision": DATASET_REVISION, "split": f"{EVAL_SPLIT or 'train'}[:1]"}
    job = json.dumps({"model": str(model), "requests": list(requests), "dataset": dataset})
    done = subprocess.run([str(PY), str(ASK_SCRIPT)], input=job, capture_output=True, text=True, cwd=WORK_DIR, env=run_environment())
    if done.returncode != 0:
        raise RuntimeError(done.stderr[-3000:])
    return json.loads(done.stdout.strip().splitlines()[-1])


# A record of the dataset, with what its labels say
result = ask(dataset_row=True)
print(json.dumps(result["answers"][0], indent=2)[:3000])
print("gold:", json.dumps(result["gold"])[:1000])

In [ ]:
# Any request of your own
REQUEST = {
    "state": "I was charged twice for my order and want a refund.",
    "questions": {
        "refund": {"type": "noul", "instructions": "Is a refund requested?"},
        "topic": {"type": "choice", "instructions": "What is it about?", "criteria": {"billing": None, "shipping": None}},
    },
}
print(json.dumps(ask(REQUEST)["answers"][0], indent=2))

## Optional: convert an older checkpoint

Set `EXISTING_CHECKPOINT` to a folder in the older layout: a backbone with `joint_head.safetensors` and
`joint_head_config.json` (the joint head, or the cosine head when its `head_type` is `cosine_span_v1`). The
converter writes a native folder and never runs code from the source. A trained head on an encoder backbone also
needs `--global_pooling` and `--prompt_style`, because the folder does not record how it was trained.

In [ ]:
if EXISTING_CHECKPOINT is not None:
    converted = WORK_DIR / "converted"
    stream([PY, "-m", "transformers.models.clef.convert_clef_weights_to_hf", "--source", EXISTING_CHECKPOINT, "--output", converted])
    print(json.dumps(ask(REQUEST, model=converted)["answers"][0], indent=2))

## Optional: time training steps

Runs the same configuration for a fixed number of steps in one process and reports records per second and the step
time, with the first steps left out.

In [ ]:
if RUN_BENCHMARK:
    steps, excluded = (30, 5) if (QUICK and ON_GPU) else (3, 1) if QUICK else (340, 20)
    stream([PY, BENCHMARK_SCRIPT, "--config", cross_entropy_config, "--steps", steps, "--warmup_steps_excluded", excluded, "--output", WORK_DIR / "benchmark.json"])
    timing = json.loads((WORK_DIR / "benchmark.json").read_text())
    for key in ("window_records_per_second", "steady_step_p50", "steady_step_p95", "peak_allocated_gib", "eval_seconds"):
        print(f"{key:28} {timing.get(key)}")
    print("backbone attention:", timing["recipe"]["attn_implementation"], "| head:", timing["recipe"]["head_type"])

## Notes

**Several GPUs.** `NUM_GPUS = None` uses every visible GPU; the global batch is split evenly, so the optimizer sees
the same number of records per step whatever the count.

| GPUs | Records per GPU | Global batch |
|---|---:|---:|
| 1 | 32 | 32 |
| 2 | 16 | 32 |
| 4 | 8 | 32 |
| 8 | 4 | 32 |

**The environment.** `clef_decision_run/env` holds the virtual environment; delete it to start again, or set
`INSTALL = False` to reuse it as it is. The flash-attention kernel has builds for torch 2.10 and newer; with an
older torch (`ENVIRONMENT = "kernel"`) the backbone falls back to the default attention.

**Kernel tuning.** One of KLM's FLA kernels is tuned again for every new token count of a batch, which costs about
a third of each step on batches it has not seen. `FLA_PRESET_DIR` points FLA at a folder of saved presets (one JSON
file per kernel, as FLA and your `auto_fla_cache.py` write them) and sets `FLA_CACHE_MODE=fuzzy`, so a tuned
configuration is reused for any token count.

**In a terminal.** Everything above is plain commands on the files in the run folder (`clef_decision_run/`):

```bash
env/bin/python -m accelerate.commands.launch --multi_gpu --num_processes 8 --mixed_precision bf16 --dynamo_backend no \
    trl/examples/decisions/clef_typed_decisions.py --config cross_entropy.yaml
```

For one GPU leave out `--multi_gpu` and use `--num_processes 1`.